Import Libraries

In [30]:
import ollama
import pandas as pd
from tqdm import tqdm
import re

Import Data

In [31]:
data = pd.read_csv('DisneylandReviews.csv', encoding='latin1')
data = data[data['Reviewer_Location'] == 'Australia'].head(1000)
data
results = []
reviewer_list = []
counter = 0

Convert file to string

In [ ]:
for reviewer in tqdm(data['Review_ID'], desc = "Processing Reviewer"):

    #Get the review for each reviewer
    review = data[data['Review_ID'] == reviewer]['Review_Text'].values[0]

    #Generate the prompt
    prompt = f"Analyze the following dataset and read the review and label the review as positive or negative. Always answer with the keyword <Answer>:\n\n{review}"

    #Get the response from the model
    response = ollama.chat(model="deepseek-r1:7b", messages=[{"role": "user", "content": prompt}])  

    #Get the reponse as text
    response_text = response["message"]["content"]  

    reviewer_list.append(reviewer)  
    # Extract text inside <think>
    thinking_match = re.search(r"<think>(.*?)</think>", response_text, re.DOTALL)
    thinking_text = thinking_match.group(1).strip() if thinking_match else "N/A"

    answer_text = "N/A"  
    if "</think>" in response_text:  
        post_think_text = response_text.split("</think>", 1)[1].strip()  

        answer_text = post_think_text

    match = re.search(r"<Answer>(.*?)</Answer>", answer_text)
    if match:
        extracted_value = match.group(1)

    results.append({"reviewer": reviewer, "thinking": thinking_text, "answer": answer_text, "positive-negative": extracted_value})

    counter+=1
    if counter == 10:
        break

Run the model and get a response

In [ ]:
output = pd.DataFrame(results)
output